# Dinámica de Poblaciones — Modelo Logístico Discreto
## Cobweb Plot: Convergencia y Oscilaciones

El **modelo logístico discreto** describe la evolución de una población $X_n$ mediante la ecuación en diferencias:

$$X_{n+1} = X_n + \rho X_n (L - X_n)$$

donde:
- $\rho > 0$ es la **tasa intrínseca de crecimiento**,
- $L > 0$ es la **capacidad de carga** (carrying capacity).

### Puntos de equilibrio

Resolviendo $X^* = X^* + \rho X^*(L - X^*)$ se obtienen dos puntos fijos:

$$X^* = 0 \qquad \text{(trivial, inestable)} \qquad \text{y} \qquad X^* = L \qquad \text{(no trivial)}$$

### Condición de estabilidad de $X^* = L$

La derivada del mapa evaluada en $X^* = L$ es:

$$f'(X) = 1 + \rho(L - 2X) \implies f'(L) = 1 - \rho L$$

El equilibrio $X^* = L$ es **localmente estable** si y solo si:

$$|1 - \rho L| < 1 \quad \Longrightarrow \quad 0 < \rho < \frac{2}{L}$$

### Cobweb plot

El **diagrama de telaraña** visualiza la dinámica iterativa:
- Se grafican $f(X) = X + \rho X(L-X)$ y la diagonal $y = X$.
- Partiendo de $X_0$, se traza una línea **vertical** hasta la curva, luego **horizontal** a la diagonal, y se repite.

**Comportamiento esperado** (con $L = 1$ por defecto):
- $\rho$ pequeño ($\rho < 2/L$): convergencia al equilibrio $X^* = L$.
- $\rho$ grande ($\rho > 2/L$): oscilaciones o caos alrededor de $X^* = L$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Estilo general ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#161b27',
    'axes.edgecolor':    '#2e3650',
    'axes.labelcolor':   '#cdd6f4',
    'xtick.color':       '#7f849c',
    'ytick.color':       '#7f849c',
    'grid.color':        '#2e3650',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'text.color':        '#cdd6f4',
    'font.family':       'monospace',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
})

In [ ]:
# ── Modelo logístico discreto ────────────────────────────────────────────────
def f(X, rho, L=1.0):
    """Mapa logístico: X_{n+1} = X_n + rho * X_n * (L - X_n)"""
    return X + rho * X * (L - X)

def iterar(X0, rho, L=1.0, pasos=40):
    """Genera la serie temporal del modelo a partir de la condición inicial X0."""
    trayectoria = [X0]
    for _ in range(pasos):
        trayectoria.append(f(trayectoria[-1], rho, L))
    return np.array(trayectoria)

def cobweb_coords(X0, rho, L=1.0, pasos=30):
    """
    Genera las coordenadas (x, y) del diagrama cobweb.
    Cada iteración produce dos segmentos:
      1. Vertical:   (X_n, X_n) -> (X_n, X_{n+1})    [hasta la curva]
      2. Horizontal: (X_n, X_{n+1}) -> (X_{n+1}, X_{n+1}) [hasta la diagonal]
    """
    xs, ys = [X0], [0.0]   # punto de arranque sobre el eje x
    X = X0
    for _ in range(pasos):
        Xf = f(X, rho, L)  # siguiente iterado (punto en la curva)
        xs += [X,  Xf]     # segmento vertical hasta la curva
        ys += [Xf, Xf]     # segmento horizontal hasta la diagonal
        X = Xf
    return np.array(xs), np.array(ys)

In [ ]:
# ── Función principal de visualización ──────────────────────────────────────
def plot_cobweb(rho, L=1.0, X0=0.1, pasos=35, titulo_extra='',
                color_tela='#89dceb', color_serie='#a6e3a1',
                ax_cobweb=None, ax_serie=None, standalone=True):
    """
    Dibuja el cobweb plot y la serie temporal para un valor de rho dado.

    Parámetros
    ----------
    rho          : tasa intrínseca de crecimiento
    L            : capacidad de carga (por defecto L=1)
    X0           : condición inicial (0 < X0 < L recomendado)
    pasos        : número de iteraciones
    titulo_extra : texto adicional en el título
    color_tela   : color de las líneas del cobweb
    color_serie  : color de la serie temporal
    standalone   : si True, crea su propia figura
    """
    umbral = 2.0 / L   # rho crítico de bifurcación

    if standalone:
        fig = plt.figure(figsize=(12, 5), facecolor='#0f1117')
        gs  = gridspec.GridSpec(1, 2, wspace=0.38)
        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1])
    else:
        ax1, ax2 = ax_cobweb, ax_serie

    # Rango de graficación: ligeramente más allá de 2L para ver la parábola
    margen  = 0.25 * L
    X_min   = -margen
    X_max   = 2.0 * L + margen
    X_vals  = np.linspace(X_min, X_max, 800)
    fX      = f(X_vals, rho, L)

    # ── Panel izquierdo: cobweb ──────────────────────────────────────────────
    label_curva = r'$f(X)=X+\rho X(L-X)$'
    ax1.plot(X_vals, fX,     color='#f38ba8', lw=2.2, label=label_curva)
    ax1.plot(X_vals, X_vals, color='#fab387', lw=1.4, ls='--',
             label=r'$y = X$ (diagonal)')

    # Líneas de referencia en los puntos fijos
    ax1.axvline(L,   color='#6c7086', lw=0.8, ls=':')
    ax1.axhline(L,   color='#6c7086', lw=0.8, ls=':')
    ax1.axvline(0.0, color='#45475a', lw=0.6, ls=':')

    # Telaraña
    xs, ys = cobweb_coords(X0, rho, L, pasos)
    # Filtrar valores fuera de rango para no distorsionar el plot
    mask = (ys > X_min - 1) & (ys < X_max + 1)
    ax1.plot(xs, ys, color=color_tela, lw=1.0, alpha=0.85, zorder=3)

    # Puntos notables
    ax1.scatter(X0, 0, color='#f9e2af', s=55, zorder=5,
                label=f'$X_0 = {X0}$')
    ax1.scatter(L, L, color='#a6e3a1', s=80, zorder=5,
                marker='*', label=f'Equilibrio $X^*=L={L}$')
    ax1.scatter(0, 0, color='#f38ba8', s=40, zorder=5,
                marker='x', label='Equilibrio trivial $X^*=0$')

    ax1.set_xlim(X_min, X_max)
    ax1.set_ylim(X_min, X_max)
    ax1.set_xlabel('$X_n$')
    ax1.set_ylabel('$X_{n+1}$')
    ax1.set_title(
        f'Cobweb  |  $\\rho={rho}$, $L={L}$  '
        f'(umbral $2/L={umbral:.2f}$) {titulo_extra}',
        pad=10, fontsize=11
    )
    ax1.legend(fontsize=7.5, loc='upper right',
               framealpha=0.2, edgecolor='#45475a')
    ax1.grid(True)

    # ── Panel derecho: serie temporal ────────────────────────────────────────
    serie = iterar(X0, rho, L, pasos)
    t     = np.arange(len(serie))

    ax2.plot(t, serie, color=color_serie, lw=1.5, marker='o',
             markersize=4, alpha=0.9)
    ax2.axhline(L, color='#fab387', lw=1.2, ls='--', alpha=0.75,
                label=f'$X^*=L={L}$')
    ax2.set_xlabel('Tiempo $n$')
    ax2.set_ylabel('$X_n$')
    ax2.set_title(
        f'Serie temporal  |  $\\rho={rho}$, $L={L}$',
        pad=10, fontsize=11
    )
    ax2.legend(fontsize=9, framealpha=0.2, edgecolor='#45475a')
    ax2.grid(True)

    if standalone:
        plt.tight_layout()
        plt.show()

---
## Parámetros globales

Fijamos la capacidad de carga $L = 1$ (puede modificarse). El umbral de inestabilidad queda en $\rho^* = 2/L = 2$.

In [ ]:
L  = 1.0            # Capacidad de carga
X0 = 0.1            # Condición inicial
print(f"Capacidad de carga:       L  = {L}")
print(f"Umbral de bifurcación:    ρ* = 2/L = {2/L}")
print(f"Condición inicial:        X0 = {X0}")

---
## Caso 1 — $\rho$ pequeño: convergencia al equilibrio

Para $\rho < 2/L$, el punto fijo $X^* = L$ es **estable**. La telaraña converge monótonamente (espiral o escalera) hacia $X^* = L$.

In [ ]:
plot_cobweb(rho=0.8, L=L, X0=X0, pasos=30,
            titulo_extra='— Convergencia estable',
            color_tela='#89dceb', color_serie='#a6e3a1')

---
## Caso 2 — $\rho$ moderado: oscilaciones amortiguadas

Cerca del umbral ($\rho = 1.8 < 2/L$), el sistema aún converge pero con **oscilaciones amortiguadas**: la telaraña se enrosca en espiral hacia $X^* = L$.

In [ ]:
plot_cobweb(rho=1.8, L=L, X0=X0, pasos=35,
            titulo_extra='— Oscilaciones amortiguadas',
            color_tela='#cba6f7', color_serie='#f9e2af')

---
## Caso 3 — $\rho$ grande: oscilaciones sostenidas / caos

Para $\rho > 2/L$, el equilibrio $X^* = L$ se vuelve **inestable**. La telaraña oscila sin converger, pasando por ciclos de período 2, 4, … hasta comportamiento caótico.

In [ ]:
plot_cobweb(rho=2.8, L=L, X0=X0, pasos=40,
            titulo_extra='— Oscilaciones / caos',
            color_tela='#f38ba8', color_serie='#fab387')

---
## Comparativa completa — Cuatro valores de $\rho$

In [ ]:
casos = [
    dict(rho=0.5,  color_tela='#89dceb', color_serie='#a6e3a1', titulo='Convergencia rápida'),
    dict(rho=1.5,  color_tela='#cba6f7', color_serie='#b4befe', titulo='Convergencia lenta / oscilatoria'),
    dict(rho=2.5,  color_tela='#f9e2af', color_serie='#fab387', titulo='Ciclo de período 2'),
    dict(rho=3.2,  color_tela='#f38ba8', color_serie='#eba0ac', titulo='Caos'),
]

fig = plt.figure(figsize=(18, 10), facecolor='#0f1117')
fig.suptitle(
    r'Modelo Logístico Discreto $X_{n+1}=X_n+\rho X_n(L-X_n)$ — Cobweb comparativo ($L=1$)',
    fontsize=13, color='#cdd6f4', y=1.01
)

outer_gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.35)

for i, caso in enumerate(casos):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=outer_gs[i], wspace=0.42)
    ax1 = fig.add_subplot(inner[0])
    ax2 = fig.add_subplot(inner[1])

    plot_cobweb(
        rho=caso['rho'], L=L, X0=X0, pasos=38,
        titulo_extra=f"— {caso['titulo']}",
        color_tela=caso['color_tela'],
        color_serie=caso['color_serie'],
        ax_cobweb=ax1, ax_serie=ax2,
        standalone=False
    )

plt.tight_layout()
plt.show()

---
## Resumen teórico

### Análisis del mapa

El mapa $f(X) = X + \rho X(L - X)$ es una **parábola** con:
- Vértice en $X_v = L/2$, con $f(L/2) = L/2 + \rho(L/2)(L/2) = L/2 + \rho L^2/4$.
- Raíces en $X = 0$ y $X = L + 1/\rho$ (interceptos con la diagonal dan los puntos fijos).

### Tabla de comportamientos ($L = 1$)

| $\rho$ | Comportamiento | Estabilidad de $X^*=L$ |
|--------|---------------|------------------------|
| $0 < \rho < 1$ | Convergencia **monótona** al equilibrio | **Estable** |
| $1 \leq \rho < 2$ | Convergencia con **oscilaciones amortiguadas** | **Estable** |
| $\rho = 2/L$ | Umbral de bifurcación | Neutralmente estable |
| $2/L < \rho \lesssim 2.45/L$ | **Ciclo de período 2** | Inestable |
| $\rho \gtrsim 2.45/L$ | Ciclos de período mayor y **caos** | Inestable |

### Derivación de la condición de estabilidad

$$f'(X) = 1 + \rho(L - 2X)$$

Evaluando en el equilibrio no trivial $X^* = L$:

$$f'(L) = 1 + \rho(L - 2L) = 1 - \rho L$$

Condición de estabilidad local:

$$|f'(L)| = |1 - \rho L| < 1 \quad \Longrightarrow \quad 0 < \rho < \frac{2}{L}$$